# 📊 AI-Powered Business Intelligence & Sales Forecasting System
### Dataset: SampleSuperstore.csv

**Author:** Sumit  
**Project:** AI-Powered BI & Sales Forecasting System  

---

## 📌 Project Overview

This notebook builds a complete end-to-end AI-powered Business Intelligence system on the Superstore retail dataset. It covers:

| Section | Description |
|---------|-------------|
| **1. Setup & Imports** | Install libraries, import dependencies |
| **2. Data Loading & Preprocessing** | Load CSV, clean, enrich with derived features |
| **3. Exploratory Data Analysis (EDA)** | KPIs, distributions, correlations |
| **4. Sales Analytics** | Category, region, segment, ship mode breakdowns |
| **5. Discount Impact Analysis** | How discounts kill profitability |
| **6. AI Sales Forecasting** | Polynomial trend + Prophet (6-month forecast) |
| **7. Profit Prediction (XGBoost)** | ML regression model for order profit |
| **8. Anomaly Detection** | Isolation Forest to flag suspicious orders |
| **9. Product Segmentation (K-Means)** | Cluster sub-categories into strategic groups |
| **10. Business Insights & Recommendations** | Key takeaways from all analyses |

---

## 1. 📦 Setup & Imports

In [ ]:
# Install required packages (run once)
import subprocess, sys
packages = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'plotly', 'scikit-learn', 'xgboost', 'joblib']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'])
print('All packages ready.')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Core
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import os

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
    print('XGBoost available ✓')
except ImportError:
    XGB_AVAILABLE = False
    print('XGBoost not found — using RandomForest as fallback')

try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
    print('Prophet available ✓')
except ImportError:
    PROPHET_AVAILABLE = False
    print('Prophet not found — using polynomial trend fallback')

# Notebook display
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 5)

print('\nAll imports successful ✓')

---
## 2. 📂 Data Loading & Preprocessing

In [ ]:
# ── Load raw CSV ──────────────────────────────────────────────────────────────
DATA_PATH = 'SampleSuperstore.csv'
df_raw = pd.read_csv(DATA_PATH)
df_raw.columns = df_raw.columns.str.strip()

print(f'Shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head()

In [ ]:
print('=== Data Types ===')
print(df_raw.dtypes)
print('\n=== Missing Values ===')
print(df_raw.isnull().sum())
print('\n=== Duplicates ===')
print(f'Duplicate rows: {df_raw.duplicated().sum()}')

In [ ]:
# ── Preprocessing ─────────────────────────────────────────────────────────────
df = df_raw.drop_duplicates().copy()
df = df.reset_index(drop=True)

# Derived numeric features
df['Profit_Margin']    = np.where(df['Sales'] != 0, df['Profit'] / df['Sales'] * 100, 0)
df['Revenue_per_Unit'] = np.where(df['Quantity'] != 0, df['Sales'] / df['Quantity'], 0)
df['Discount_Impact']  = df['Sales'] * df['Discount']

# Synthetic Order Dates: Jan-2021 → Dec-2023 (for time-series analysis)
n = len(df)
df['Order_Date'] = pd.date_range('2021-01-01', '2023-12-31', periods=n)
df['Year']       = df['Order_Date'].dt.year
df['Month']      = df['Order_Date'].dt.month
df['Month_Name'] = df['Order_Date'].dt.strftime('%b')
df['Quarter']    = df['Order_Date'].dt.quarter
df['Week']       = df['Order_Date'].dt.isocalendar().week.astype(int)
df['Day_of_Week']= df['Order_Date'].dt.day_name()

print(f'Cleaned dataset shape: {df.shape}')
df[['Sales','Profit','Quantity','Discount','Profit_Margin','Revenue_per_Unit']].describe()

---
## 3. 🔍 Exploratory Data Analysis (EDA)

In [ ]:
# ── Top-Level KPIs ────────────────────────────────────────────────────────────
kpis = {
    'Total Sales ($)':          df['Sales'].sum(),
    'Total Profit ($)':         df['Profit'].sum(),
    'Total Orders':             len(df),
    'Avg Profit Margin (%)':    df['Profit_Margin'].mean(),
    'Total Units Sold':         df['Quantity'].sum(),
    'Avg Discount (%)':         df['Discount'].mean() * 100,
    'Cities Served':            df['City'].nunique(),
    'Avg Order Value ($)':      df['Sales'].mean(),
}

print('=' * 45)
print('         BUSINESS KPI DASHBOARD')
print('=' * 45)
for k, v in kpis.items():
    if '($)' in k:
        print(f'  {k:<30} ${v:>12,.2f}')
    elif '(%)' in k:
        print(f'  {k:<30} {v:>12.2f}%')
    else:
        print(f'  {k:<30} {v:>12,.0f}')
print('=' * 45)

In [ ]:
# ── Distribution Plots ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].hist(df['Sales'], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0,0].set_title('Sales Distribution', fontsize=13, fontweight='bold')
axes[0,0].set_xlabel('Sales ($)')
axes[0,0].set_ylabel('Frequency')
axes[0,0].axvline(df['Sales'].median(), color='red', linestyle='--', label=f'Median: ${df["Sales"].median():,.0f}')
axes[0,0].legend()

axes[0,1].hist(df['Profit'], bins=60, color='seagreen', edgecolor='white', alpha=0.85)
axes[0,1].set_title('Profit Distribution', fontsize=13, fontweight='bold')
axes[0,1].set_xlabel('Profit ($)')
axes[0,1].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Break-even')
axes[0,1].legend()

axes[1,0].hist(df['Profit_Margin'], bins=60, color='darkorange', edgecolor='white', alpha=0.85)
axes[1,0].set_title('Profit Margin Distribution (%)', fontsize=13, fontweight='bold')
axes[1,0].set_xlabel('Profit Margin (%)')
axes[1,0].axvline(0, color='red', linestyle='--', linewidth=1.5)

axes[1,1].hist(df['Discount'], bins=20, color='mediumpurple', edgecolor='white', alpha=0.85)
axes[1,1].set_title('Discount Distribution', fontsize=13, fontweight='bold')
axes[1,1].set_xlabel('Discount Rate')

plt.suptitle('Feature Distributions — SampleSuperstore', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation Heatmap ───────────────────────────────────────────────────────
numeric_cols = ['Sales', 'Quantity', 'Discount', 'Profit', 'Profit_Margin', 'Revenue_per_Unit', 'Discount_Impact']
corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', mask=mask,
            linewidths=0.5, vmin=-1, vmax=1, center=0,
            annot_kws={'size': 11})
plt.title('Correlation Matrix — Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('Key insight: Discount has a strong NEGATIVE correlation with Profit_Margin')

---
## 4. 📊 Sales Analytics

In [ ]:
# ── Category Summary ──────────────────────────────────────────────────────────
cat_df = df.groupby('Category').agg(
    Sales=('Sales','sum'), Profit=('Profit','sum'),
    Quantity=('Quantity','sum'), Orders=('Sales','count')
).reset_index().sort_values('Sales', ascending=False)
cat_df['Profit_Margin_%'] = (cat_df['Profit'] / cat_df['Sales'] * 100).round(2)

print('=== Category Performance ===')
print(cat_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#3b82d4', '#22c55e', '#f59e0b']
bars = axes[0].bar(cat_df['Category'], cat_df['Sales'], color=colors, edgecolor='white', alpha=0.9)
axes[0].set_title('Total Sales by Category', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Sales ($)')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
for bar, val in zip(bars, cat_df['Sales']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+500, f'${val/1000:.1f}K',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

bars2 = axes[1].bar(cat_df['Category'], cat_df['Profit'], color=colors, edgecolor='white', alpha=0.9)
axes[1].set_title('Total Profit by Category', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Profit ($)')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
for bar, val in zip(bars2, cat_df['Profit']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
                f'${val/1000:.1f}K', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Sub-Category Analysis ─────────────────────────────────────────────────────
sub_df = df.groupby(['Category','Sub-Category']).agg(
    Sales=('Sales','sum'), Profit=('Profit','sum'), Quantity=('Quantity','sum')
).reset_index().sort_values('Sales', ascending=False)
sub_df['Profit_Margin_%'] = (sub_df['Profit'] / sub_df['Sales'] * 100).round(1)

top15 = sub_df.sort_values('Sales', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(12, 8))
colors_bar = ['#22c55e' if p >= 0 else '#ef4444' for p in top15['Profit']]
bars = ax.barh(top15['Sub-Category'], top15['Sales'], color='steelblue', alpha=0.85)
ax.set_title('Top 15 Sub-Categories by Sales', fontsize=14, fontweight='bold')
ax.set_xlabel('Sales ($)')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
plt.tight_layout()
plt.show()

# Profit margin by sub-category (worst performers)
worst = sub_df.sort_values('Profit_Margin_%').head(8)
print('\n=== Most Loss-Making Sub-Categories ===')
print(worst[['Category','Sub-Category','Sales','Profit','Profit_Margin_%']].to_string(index=False))

In [ ]:
# ── Regional & Segment Analysis ───────────────────────────────────────────────
region_df = df.groupby('Region').agg(
    Sales=('Sales','sum'), Profit=('Profit','sum'), Orders=('Sales','count')
).reset_index().sort_values('Sales', ascending=False)
region_df['Profit_Margin_%'] = (region_df['Profit']/region_df['Sales']*100).round(1)

seg_df = df.groupby('Segment').agg(
    Sales=('Sales','sum'), Profit=('Profit','sum'), Orders=('Sales','count')
).reset_index().sort_values('Sales', ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Region pie
axes[0].pie(region_df['Sales'], labels=region_df['Region'], autopct='%1.1f%%',
           startangle=140, colors=sns.color_palette('Set2', len(region_df)))
axes[0].set_title('Sales by Region', fontsize=13, fontweight='bold')

# Segment bar
axes[1].bar(seg_df['Segment'], seg_df['Sales'],
           color=sns.color_palette('Set2', len(seg_df)), edgecolor='white', alpha=0.9)
axes[1].set_title('Sales by Segment', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Sales ($)')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))

# Ship mode bar
ship_df = df.groupby('Ship Mode').agg(Orders=('Sales','count')).reset_index().sort_values('Orders',ascending=False)
axes[2].bar(ship_df['Ship Mode'], ship_df['Orders'],
           color=sns.color_palette('Set2', len(ship_df)), edgecolor='white', alpha=0.9)
axes[2].set_title('Orders by Ship Mode', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Number of Orders')
plt.setp(axes[2].get_xticklabels(), rotation=15, ha='right')

plt.tight_layout()
plt.show()

print('\n=== Regional Summary ===')
print(region_df.to_string(index=False))

In [ ]:
# ── Top 10 States ─────────────────────────────────────────────────────────────
state_df = df.groupby('State').agg(
    Sales=('Sales','sum'), Profit=('Profit','sum'), Orders=('Sales','count')
).reset_index().sort_values('Sales', ascending=False)

top10 = state_df.head(10).sort_values('Sales', ascending=True)
colors_state = ['#ef4444' if p < 0 else '#3b82d4' for p in top10['Profit']]

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top10['State'], top10['Sales'], color=colors_state, edgecolor='white', alpha=0.9)
ax.set_title('Top 10 States by Sales  (red = net loss state)', fontsize=13, fontweight='bold')
ax.set_xlabel('Sales ($)')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
plt.tight_layout()
plt.show()

---
## 5. 💸 Discount Impact Analysis

In [ ]:
# ── Scatter: Sales vs Profit coloured by Discount ────────────────────────────
sample = df.sample(min(600, len(df)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sc = axes[0].scatter(sample['Sales'], sample['Profit'],
                     c=sample['Discount'], cmap='RdYlGn_r',
                     alpha=0.65, s=sample['Quantity']*8, edgecolors='none')
plt.colorbar(sc, ax=axes[0], label='Discount Rate')
axes[0].axhline(0, color='grey', linestyle='--', linewidth=1)
axes[0].set_title('Sales vs Profit  (color=Discount, size=Qty)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Sales ($)')
axes[0].set_ylabel('Profit ($)')

# Bucket discount and show avg profit
bins   = [-0.01, 0, 0.1, 0.2, 0.3, 0.5, 0.8, 1.0]
labels = ['0%','1-10%','11-20%','21-30%','31-50%','51-80%','>80%']
df_disc = df.copy()
df_disc['Discount_Bucket'] = pd.cut(df_disc['Discount'], bins=bins, labels=labels)
disc_agg = df_disc.groupby('Discount_Bucket', observed=True).agg(
    Avg_Profit=('Profit','mean'), Count=('Profit','count')
).reset_index()

bar_colors = ['#22c55e' if p >= 0 else '#ef4444' for p in disc_agg['Avg_Profit']]
axes[1].bar(disc_agg['Discount_Bucket'], disc_agg['Avg_Profit'],
           color=bar_colors, edgecolor='white', alpha=0.9)
axes[1].axhline(0, color='grey', linestyle='--', linewidth=1)
axes[1].set_title('Avg Profit by Discount Bucket', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Discount Level')
axes[1].set_ylabel('Avg Profit ($)')

plt.tight_layout()
plt.show()

print('\n=== Discount vs Avg Profit ===')
print(disc_agg.to_string(index=False))
print('\n⚠️  KEY INSIGHT: Discounts > 30% result in NEGATIVE average profit!')

---
## 6. 📈 AI Sales Forecasting

In [ ]:
# ── Aggregate to monthly time series ─────────────────────────────────────────
monthly = df.resample('ME', on='Order_Date').agg(
    Sales=('Sales','sum'), Profit=('Profit','sum'),
    Quantity=('Quantity','sum'), Orders=('Sales','count')
).reset_index()
monthly.rename(columns={'Order_Date':'ds'}, inplace=True)
monthly['y'] = monthly['Sales']

print(f'Monthly time series: {len(monthly)} periods')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly['ds'], monthly['Sales'], color='steelblue', linewidth=2, marker='o', markersize=4, label='Monthly Sales')
ax.plot(monthly['ds'], monthly['Profit'], color='seagreen', linewidth=1.5, linestyle='--', label='Monthly Profit')
ax.fill_between(monthly['ds'], monthly['Sales'], alpha=0.08, color='steelblue')
ax.set_title('Monthly Sales & Profit Trend (Jan 2021 – Dec 2023)', fontsize=13, fontweight='bold')
ax.set_ylabel('USD ($)')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Forecasting: Prophet (if available) else Polynomial Trend ─────────────────
FORECAST_PERIODS = 6

if PROPHET_AVAILABLE:
    print('Running Prophet forecast...')
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.1,
    )
    model.add_seasonality(name='quarterly', period=91.25, fourier_order=5)
    model.fit(monthly[['ds','y']])
    future = model.make_future_dataframe(periods=FORECAST_PERIODS, freq='ME')
    forecast = model.predict(future)
    fc_ds    = forecast['ds']
    fc_yhat  = forecast['yhat']
    fc_lower = forecast['yhat_lower']
    fc_upper = forecast['yhat_upper']
    method = 'Prophet'

else:
    print('Running polynomial trend forecast...')
    t = np.arange(len(monthly))
    coeffs = np.polyfit(t, monthly['y'], deg=2)
    poly   = np.poly1d(coeffs)
    future_t     = np.arange(len(monthly), len(monthly) + FORECAST_PERIODS)
    future_dates = pd.date_range(monthly['ds'].iloc[-1] + pd.DateOffset(months=1),
                                  periods=FORECAST_PERIODS, freq='ME')
    fut_vals  = poly(future_t)
    resid_std = (monthly['y'] - poly(t)).std()

    history_fc = pd.DataFrame({'ds': monthly['ds'], 'yhat': poly(t),
                                'yhat_lower': poly(t) - 1.96*resid_std,
                                'yhat_upper': poly(t) + 1.96*resid_std})
    future_fc  = pd.DataFrame({'ds': future_dates, 'yhat': fut_vals,
                                'yhat_lower': fut_vals - 1.96*resid_std,
                                'yhat_upper': fut_vals + 1.96*resid_std})
    forecast   = pd.concat([history_fc, future_fc], ignore_index=True)
    fc_ds    = forecast['ds']
    fc_yhat  = forecast['yhat']
    fc_lower = forecast['yhat_lower']
    fc_upper = forecast['yhat_upper']
    method = 'Polynomial Trend'

print(f'Forecast method: {method}')

In [ ]:
# ── Plot Forecast ─────────────────────────────────────────────────────────────
cutoff = monthly['ds'].max()
fut_mask = fc_ds > cutoff

fig, ax = plt.subplots(figsize=(14, 6))

# Confidence band for forecast
ax.fill_between(fc_ds[fut_mask], fc_lower[fut_mask], fc_upper[fut_mask],
                alpha=0.2, color='tomato', label='95% Confidence Interval')

# Historical actuals
ax.plot(monthly['ds'], monthly['y'], 'o-', color='steelblue', linewidth=2,
        markersize=4, label='Actual Sales')

# Fitted trend
ax.plot(fc_ds[~fut_mask], fc_yhat[~fut_mask], '--', color='orange',
        linewidth=1.5, label='Fitted Trend')

# Future forecast
ax.plot(fc_ds[fut_mask], fc_yhat[fut_mask], 'D-', color='tomato',
        linewidth=2.5, markersize=7, label=f'Forecast ({method})')

# Vertical divider
ax.axvline(cutoff, color='grey', linestyle='--', linewidth=1.5, label='Forecast Start')
ax.set_title(f'AI Sales Forecast — {method}  |  Horizon: {FORECAST_PERIODS} months',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Sales (USD)')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

# Print forecast table
future_rows = forecast[forecast['ds'] > cutoff][['ds','yhat','yhat_lower','yhat_upper']].copy()
future_rows.columns = ['Month','Forecast ($)','Lower Bound ($)','Upper Bound ($)']
future_rows['Month'] = future_rows['Month'].dt.strftime('%b %Y')
print('\n=== 6-Month Sales Forecast ===')
print(future_rows.to_string(index=False))

---
## 7. 🤖 Profit Prediction (XGBoost)

In [ ]:
# ── Feature Engineering for ML ────────────────────────────────────────────────
feat_cols  = ['Sales','Quantity','Discount','Category','Sub-Category',
              'Region','Segment','Ship Mode','Month','Quarter']
target_col = 'Profit'

ml_data = df[feat_cols + [target_col]].copy().dropna()

cat_cols = ['Category','Sub-Category','Region','Segment','Ship Mode']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    ml_data[col] = le.fit_transform(ml_data[col].astype(str))
    encoders[col] = le

X = ml_data[feat_cols]
y = ml_data[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape[0]} rows  |  Test: {X_test.shape[0]} rows')
print(f'Features: {list(X.columns)}')

In [ ]:
# ── Train Model ───────────────────────────────────────────────────────────────
if XGB_AVAILABLE:
    model_profit = XGBRegressor(
        n_estimators=300, max_depth=6,
        learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, random_state=42, verbosity=0
    )
    model_name = 'XGBoost'
else:
    model_profit = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    model_name = 'RandomForest'

print(f'Training {model_name} model...')
model_profit.fit(X_train, y_train)
preds = model_profit.predict(X_test)

mae  = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2   = r2_score(y_test, preds)

print(f'\n=== {model_name} Model Performance ===')
print(f'  MAE  (Mean Absolute Error):  ${mae:,.2f}')
print(f'  RMSE (Root Mean Sq Error):   ${rmse:,.2f}')
print(f'  R²   (Coefficient of Det.):  {r2:.4f}')

# Save model
os.makedirs('models', exist_ok=True)
joblib.dump({'model': model_profit, 'encoders': encoders}, 'models/profit_model.pkl')
print('\nModel saved to models/profit_model.pkl')

In [ ]:
# ── Actual vs Predicted + Feature Importance ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Actual vs Predicted
axes[0].scatter(y_test, preds, alpha=0.4, color='steelblue', s=15)
mn = min(y_test.min(), preds.min())
mx = max(y_test.max(), preds.max())
axes[0].plot([mn,mx],[mn,mx],'r--', linewidth=1.5, label='Perfect Fit')
axes[0].set_xlabel('Actual Profit ($)')
axes[0].set_ylabel('Predicted Profit ($)')
axes[0].set_title(f'{model_name}: Actual vs Predicted Profit\nR²={r2:.4f}  MAE=${mae:,.1f}',
                  fontsize=12, fontweight='bold')
axes[0].legend()

# Feature Importance
feat_imp = pd.DataFrame({'Feature': X.columns, 'Importance': model_profit.feature_importances_})\
             .sort_values('Importance', ascending=True).tail(10)
axes[1].barh(feat_imp['Feature'], feat_imp['Importance'],
            color='steelblue', alpha=0.85, edgecolor='white')
axes[1].set_title('Feature Importances — Profit Predictor', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()

In [ ]:
# ── Interactive Profit Prediction Demo ───────────────────────────────────────
def predict_profit(sales, quantity, discount, category, subcategory,
                   region, segment, ship_mode, month, quarter):
    row = {
        'Sales': sales, 'Quantity': quantity, 'Discount': discount,
        'Category':    encoders['Category'].transform([category])[0],
        'Sub-Category':encoders['Sub-Category'].transform([subcategory])[0],
        'Region':      encoders['Region'].transform([region])[0],
        'Segment':     encoders['Segment'].transform([segment])[0],
        'Ship Mode':   encoders['Ship Mode'].transform([ship_mode])[0],
        'Month': month, 'Quarter': quarter,
    }
    return float(model_profit.predict(pd.DataFrame([row]))[0])

# Example predictions
examples = [
    (500,  3, 0.0,  'Technology',      'Phones',     'West',    'Consumer',   'Standard Class', 6, 2),
    (1000, 5, 0.45, 'Furniture',       'Tables',     'South',   'Consumer',   'Standard Class', 3, 1),
    (200,  2, 0.2,  'Office Supplies', 'Binders',    'East',    'Corporate',  'Second Class',   9, 3),
    (3000, 7, 0.0,  'Technology',      'Copiers',    'West',    'Corporate',  'First Class',    11, 4),
]

print(f'=== Profit Prediction Demo ({model_name}) ===')
print(f'{"Category":<20} {"Sub-Cat":<15} {"Sales":>8} {"Disc":>6} {"Pred Profit":>12} {"Margin":>8}')
print('-'*75)
for (s,q,d,cat,sub,reg,seg,sm,mo,qtr) in examples:
    pred = predict_profit(s,q,d,cat,sub,reg,seg,sm,mo,qtr)
    margin = pred/s*100
    flag = '⚠️' if pred < 0 else '✅'
    print(f'{cat:<20} {sub:<15} {s:>8,.0f} {d:>6.0%} {pred:>12,.2f} {margin:>7.1f}%  {flag}')

---
## 8. 🔍 Anomaly Detection (Isolation Forest)

In [ ]:
# ── Isolation Forest ─────────────────────────────────────────────────────────
CONTAMINATION = 0.05  # expect 5% anomalies

features_iso = df[['Sales','Profit','Quantity','Discount']].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features_iso)

iso = IsolationForest(contamination=CONTAMINATION, random_state=42, n_jobs=-1)
df['Anomaly']       = iso.fit_predict(X_scaled)
df['Anomaly_Score'] = iso.decision_function(X_scaled)
df['Is_Anomaly']    = df['Anomaly'] == -1

n_anom = df['Is_Anomaly'].sum()
print(f'Anomalous orders detected: {n_anom} ({n_anom/len(df)*100:.1f}%)')

In [ ]:
# ── Visualise Anomalies ───────────────────────────────────────────────────────
normal = df[~df['Is_Anomaly']]
anomalies = df[df['Is_Anomaly']]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Scatter: Sales vs Profit
axes[0].scatter(normal['Sales'], normal['Profit'],
               c='steelblue', alpha=0.35, s=12, label='Normal')
axes[0].scatter(anomalies['Sales'], anomalies['Profit'],
               c='tomato', alpha=0.85, s=40, marker='x', linewidths=1.5, label='Anomaly')
axes[0].axhline(0, color='grey', linestyle='--', linewidth=1)
axes[0].set_title('Anomaly Detection — Sales vs Profit', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Sales ($)')
axes[0].set_ylabel('Profit ($)')
axes[0].legend()

# Anomaly score distribution
axes[1].hist(normal['Anomaly_Score'], bins=40, color='steelblue', alpha=0.7, label='Normal')
axes[1].hist(anomalies['Anomaly_Score'], bins=20, color='tomato', alpha=0.7, label='Anomaly')
axes[1].axvline(0, color='black', linestyle='--', linewidth=1.5, label='Boundary')
axes[1].set_title('Anomaly Score Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Anomaly Score (lower = more anomalous)')
axes[1].legend()

plt.tight_layout()
plt.show()

print('\n=== Top 10 Most Anomalous Orders ===')
top_anom = anomalies.sort_values('Anomaly_Score')[['Category','Sub-Category','Region','Sales','Profit','Discount','Anomaly_Score']].head(10)
print(top_anom.to_string(index=False))

---
## 9. 📦 Product Segmentation (K-Means Clustering)

In [ ]:
# ── K-Means on Sub-Category Level ─────────────────────────────────────────────
N_CLUSTERS = 4

agg_sub = df.groupby('Sub-Category').agg(
    Avg_Sales=('Sales','mean'), Total_Sales=('Sales','sum'),
    Avg_Profit=('Profit','mean'), Avg_Discount=('Discount','mean'),
    Total_Qty=('Quantity','sum')
).reset_index()

feat_km = agg_sub[['Avg_Sales','Avg_Profit','Avg_Discount','Total_Qty']].fillna(0)
scaler_km = StandardScaler()
X_km = scaler_km.fit_transform(feat_km)

# Elbow method
inertias = []
k_range = range(2, 9)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_km)
    inertias.append(km.inertia_)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].plot(list(k_range), inertias, 'o-', color='steelblue', linewidth=2, markersize=7)
axes[0].axvline(N_CLUSTERS, color='tomato', linestyle='--', label=f'Chosen k={N_CLUSTERS}')
axes[0].set_title('Elbow Method — Optimal K', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].legend()

# Fit with chosen k
km_final = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
agg_sub['Cluster'] = km_final.fit_predict(X_km)

cluster_labels = {0:'Low Value', 1:'High Volume', 2:'High Profit', 3:'High Discount Risk'}
agg_sub['Cluster_Label'] = agg_sub['Cluster'].map(cluster_labels).fillna('Cluster ' + agg_sub['Cluster'].astype(str))

colors_cl = {'Low Value':'#94a3b8','High Volume':'#3b82d4','High Profit':'#22c55e','High Discount Risk':'#ef4444'}
for label, grp in agg_sub.groupby('Cluster_Label'):
    axes[1].scatter(grp['Total_Sales'], grp['Avg_Profit'],
                   label=label, s=grp['Total_Qty']/50,
                   alpha=0.8, color=colors_cl.get(label,'grey'))
    for _, row in grp.iterrows():
        axes[1].annotate(row['Sub-Category'], (row['Total_Sales'], row['Avg_Profit']),
                        fontsize=7, alpha=0.75, xytext=(3,3), textcoords='offset points')

axes[1].axhline(0, color='grey', linestyle='--', linewidth=1)
axes[1].set_title('Product Segments (K-Means, bubble=Qty)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Total Sales ($)')
axes[1].set_ylabel('Avg Profit ($)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print('\n=== Product Cluster Summary ===')
print(agg_sub[['Sub-Category','Cluster_Label','Total_Sales','Avg_Profit','Avg_Discount']]
      .sort_values('Total_Sales', ascending=False).to_string(index=False))

---
## 10. 💡 Business Insights & Recommendations

In [ ]:
# ── Summary Dashboard ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('AI-Powered BI System — Executive Summary Dashboard',
             fontsize=16, fontweight='bold', y=1.01)

# 1. Category Sales
axes[0,0].bar(cat_df['Category'], cat_df['Sales'],
              color=['#3b82d4','#22c55e','#f59e0b'], edgecolor='white', alpha=0.9)
axes[0,0].set_title('Sales by Category')
axes[0,0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))

# 2. Region Pie
axes[0,1].pie(region_df['Sales'], labels=region_df['Region'], autopct='%1.1f%%',
              startangle=140, colors=sns.color_palette('Set2',4))
axes[0,1].set_title('Sales by Region')

# 3. Discount vs Avg Profit
bar_c = ['#22c55e' if p>=0 else '#ef4444' for p in disc_agg['Avg_Profit']]
axes[0,2].bar(disc_agg['Discount_Bucket'], disc_agg['Avg_Profit'],
              color=bar_c, edgecolor='white', alpha=0.9)
axes[0,2].axhline(0, color='grey', linestyle='--')
axes[0,2].set_title('Discount Level vs Avg Profit')
axes[0,2].set_ylabel('Avg Profit ($)')
plt.setp(axes[0,2].get_xticklabels(), rotation=25, ha='right')

# 4. Monthly Sales Trend
axes[1,0].plot(monthly['ds'], monthly['Sales'], color='steelblue', linewidth=2)
axes[1,0].fill_between(monthly['ds'], monthly['Sales'], alpha=0.1, color='steelblue')
axes[1,0].set_title('Monthly Sales Trend')
axes[1,0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
axes[1,0].tick_params(axis='x', rotation=25)

# 5. Feature Importance
axes[1,1].barh(feat_imp['Feature'], feat_imp['Importance'],
               color='steelblue', alpha=0.85, edgecolor='white')
axes[1,1].set_title(f'Feature Importance ({model_name})')

# 6. Anomaly Detection
axes[1,2].scatter(normal['Sales'], normal['Profit'],
                  c='steelblue', alpha=0.2, s=8, label='Normal')
axes[1,2].scatter(anomalies['Sales'], anomalies['Profit'],
                  c='tomato', alpha=0.8, s=20, marker='x', label='Anomaly')
axes[1,2].axhline(0, color='grey', linestyle='--', linewidth=1)
axes[1,2].set_title('Anomaly Detection')
axes[1,2].set_xlabel('Sales ($)')
axes[1,2].set_ylabel('Profit ($)')
axes[1,2].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── Final Recommendations ─────────────────────────────────────────────────────
print('=' * 65)
print('    AI-POWERED BI SYSTEM — KEY FINDINGS & RECOMMENDATIONS')
print('=' * 65)

total_sales  = df['Sales'].sum()
total_profit = df['Profit'].sum()
avg_margin   = df['Profit_Margin'].mean()
neg_profit   = (df['Profit'] < 0).sum()
neg_pct      = neg_profit / len(df) * 100
high_disc    = (df['Discount'] > 0.3).sum()
high_disc_pct= high_disc / len(df) * 100

print(f"""
📊 EXECUTIVE SUMMARY
  Total Revenue :  ${total_sales:>12,.2f}
  Total Profit  :  ${total_profit:>12,.2f}
  Avg Margin    :  {avg_margin:>11.2f}%
  Loss Orders   :  {neg_profit:>10,} ({neg_pct:.1f}% of all orders)
  High Disc Ord :  {high_disc:>10,} ({high_disc_pct:.1f}% — discount > 30%)
  Anomalies     :  {n_anom:>10,}
  Model R²      :  {r2:>11.4f}  (profit predictor)

🔑 KEY INSIGHTS
  1. DISCOUNT POLICY: {high_disc_pct:.0f}% of orders carry discounts >30%.
     These orders generate NEGATIVE average profit.
     → Cap discounts at 20% for Furniture & Tables.

  2. FURNITURE LOSSES: Furniture sub-categories (Tables, Bookcases)
     are the biggest profit drains despite high sales volumes.
     → Review pricing and reduce deep discounting.

  3. TECHNOLOGY = STAR: Technology has the best profit margins.
     → Prioritise upselling Phones, Copiers, Accessories.

  4. WEST REGION dominates sales. South underperforms.
     → Investigate South region for operational improvements.

  5. ANOMALIES: {n_anom} orders flagged by Isolation Forest.
     → Review for potential fraud, data errors, or pricing mistakes.

  6. FORECAST: Sales trend is upward with seasonal peaks in Q4.
     → Increase inventory and staffing in Sep–Dec.

  7. CONSUMER SEGMENT drives the highest order volume.
     → Loyalty programs could significantly boost repeat revenue.
""")
print('=' * 65)

In [ ]:
print('✅ Notebook execution complete.')
print('   Streamlit dashboard: run  →  streamlit run app.py')
print('   Saved model         : models/profit_model.pkl')